# 모의문제 4 · 원격 작업 로봇의 센서 구간별 상태 분류

**난이도 중 · 권장 30분 · Python3 · Jupyter Notebook · GPU 없음**

## 1. 상황과 목표

기지 시설 점검을 지원하는 원격 작업 로봇이 물체를 잡은 채 이동하는 시험을 수행했습니다. 운용자는 기록된 센서 구간을 보고 작업 상태와 동작 이상 유형을 확인하려고 합니다. 각 구간의 힘·토크 기록을 분석하여 다섯 가지 상태 중 하나를 분류하세요.

공개된 민간 산업용 로봇 실험 자료에 비전투 시설 점검 배경을 부여한 연습 문제입니다. 실제 군용 MUM-T 통합 운용 데이터는 아닙니다. 이미 관측된 구간의 정상·이상 유형을 진단하며, 미래 고장을 예측하거나 로봇의 제어 명령을 생성하는 과제가 아닙니다.

## 2. 제공 자료

| 파일 | 내용 |
|---|---|
| problem.ipynb | 이 지문과 수정 가능한 초기 코드 |
| train_series.csv | 학습 120개 작업 구간 × 15시점 = 1,800행 |
| train_labels.csv | 학습 구간별 정답 120행 |
| test_series.csv | 테스트 41개 작업 구간 × 15시점 = 615행 |
| sample_submission.csv | 테스트 41개 ID의 제출 형식 예시 |

각 작업 구간은 15시점 × 6채널입니다. 결측은 없고 UTF-8 BOM CSV로 저장했습니다. `sample_submission.csv`의 0은 자리표시자입니다.

| 열 | 의미 |
|---|---|
| sample_id | 작업 구간 ID. 정답 연결용이며 모델 입력에서 제외 |
| step | 구간 안의 순서 0~14. 동일 구간에서는 등간격 |
| force_x, force_y, force_z | 세 축의 힘 센서 측정값 |
| torque_x, torque_y, torque_z | 세 축의 토크 센서 측정값 |
| label | 구간 전체의 정답. train_labels.csv에만 존재 |

센서 값은 원본 정수 스케일입니다. 제공 자료만으로 정확한 물리 단위를 확정하지 않았으므로 N 또는 N·m로 가정하지 마세요. 음수는 해당 축의 반대 방향을 나타내며 결측 표시가 아닙니다. `step`은 초 단위 시간이 아닙니다. 서로 다른 sample_id 사이에는 연속 시간 관계가 없습니다.

## 3. 정답 라벨

| label | 상태 | 해석 |
|---:|---|---|
| 0 | 정상 | 정상 동작 구간 |
| 1 | 공구 충돌 | 공구 쪽에서 충돌한 사례 |
| 2 | 파지 물체 충돌 | 로봇이 잡고 있는 물체 쪽에서 충돌한 사례 |
| 3 | 하부 진행 방해 | 아래쪽 진행이 방해된 사례 |
| 4 | 하부 충돌 | 아래쪽 충돌 사례 |

센서 축의 특정 값 하나가 곧 정답을 의미하지는 않습니다. 학습 사례의 패턴을 이용하여 분류하세요.

## 4. 수행 과제

1. 작업 구간 수·길이·정답 분포를 확인합니다.
2. 구간별로 모델 입력을 구성합니다. 초기 코드는 채널별 평균 6개를 사용합니다.
3. 작업 구간 단위로 학습/검증을 나누고 초기 모델의 Macro-F1을 확인합니다.
4. 변동성·범위·시간 변화 등 특징 또는 모델을 한두 가지 개선합니다.
5. 설정을 정한 뒤 학습 구간 전체로 재학습하고 테스트 41개 구간을 예측합니다.

데이터가 작은 만큼 복잡한 딥러닝은 필요 없습니다. 한 구간에서 계산한 평균·표준편차·최솟값·최댓값·처음과 마지막의 차이 등을 시도할 수 있습니다. 15×6 값을 시간순으로 펼친 입력도 허용합니다. 별도 보고서나 가설 검정 결과는 제출하지 않습니다.

## 5. 검증 및 누출 방지

**분리 단위는 CSV의 행이 아니라 sample_id입니다.** 같은 구간의 15개 행을 학습과 검증에 나누면 안 됩니다. 서로 독립적인 작업 구간을 분류하므로, 구간 단위 계층 무작위 분리가 가능합니다. 3번의 연속 미래 예측 문제와 분리 방식이 다릅니다.

- 초기 코드는 구간을 배열로 묶은 뒤 `stratify=y`로 분리합니다.
- 센서 특징은 해당 구간의 15시점을 모두 사용해도 됩니다. 분류 시 구간 전체가 제공되는 문제입니다.
- 스케일러·특징 선택 등 학습이 필요한 처리는 검증을 분리한 뒤 학습 부분에만 맞추세요.
- sample_id, 파일 행 순서, 정답은 모델 입력에 넣지 않습니다.
- 초기 검증에서 일부 클래스는 몇 사례뿐이므로 점수 변동이 큽니다. 좋은 점수를 찾으려고 분리 난수만 반복 변경하지 마세요.

## 6. 평가 및 제출

지표는 **5개 클래스의 Macro-F1**입니다. 각 클래스를 양성으로 본 F1의 산술평균이며 0~1 사이, 높을수록 좋습니다. 계산 대상 클래스는 항상 `[0,1,2,3,4]`, 분모가 0인 클래스 점수는 0입니다.

`F1_c = 2TP_c / (2TP_c + FP_c + FN_c)`

`Macro-F1 = (F1_0 + F1_1 + F1_2 + F1_3 + F1_4) / 5`

제출은 **이 Notebook과 같은 폴더의 submission.csv**입니다. 열 이름과 순서는 `sample_id,label`입니다. 테스트의 모든 ID를 중복·누락 없이 한 번씩 포함하고 label은 정수 0~4로 저장하세요. 확률·소수·인덱스 열은 제출하지 않습니다. 채점은 ID로 연결하므로 행 순서는 무관합니다.

Notebook에도 최종 코드와 실행 결과를 남기고 저장하세요. 마지막 형식 검사 셀은 파일 구조만 확인하며 점수를 알려주지 않습니다. 로컬 연습에는 실제 시험 사이트의 최종 제출 버튼이 없으므로, 30분 종료 전에 CSV와 Notebook 저장을 완료하는 것으로 제출 절차를 연습합니다.

## 7. 시간 및 실행 환경

권장 배분: 데이터 확인 5분 → 초기 모델·검증 7분 → 특징/모델 개선 12분 → 최종 예측·저장·검사 6분.

Python3, NumPy, pandas, scikit-learn을 사용합니다. Jupyter에서 problem.ipynb를 열고 커널을 Python3로 선택하세요. 필요한 패키지 설치와 압축 해제는 연습 시작 전에 완료합니다. 코드의 `DATA_DIR = Path(".")`가 이 Notebook의 폴더를 가리키는지 확인하세요.

초기 코드는 평균 특징과 간단한 결정트리로 제출까지 실행할 수 있는 예시입니다. 그대로 제출하는 것보다 검증 성능을 비교하고 개선하는 과정을 연습하세요. 문법·패키지 문서 검색은 허용하되, 원본 정답 검색·외부 데이터 결합·출제자 자료 열람은 풀이 중 하지 않습니다. 실제 시험의 AI 도구 이용 가능 여부는 별도 규정을 따릅니다.

## 8. 풀이 후 자가 채점

별도 출제자용 ZIP을 풀고 score.py와 answer.csv를 같은 폴더에 둡니다. 해당 폴더에서 자신의 제출 파일 경로를 전달하세요.

```bash
python score.py ../problem_04/submission.csv
```

Windows에서 필요하면 `python` 대신 `py`를 사용하세요. `macro_f1`이 최종 점수입니다. 채점기는 Python 표준 라이브러리만 필요합니다. 출처·라이선스·가공 내역·해설은 출제자용 자료에만 기록했습니다.


## 데이터 읽기와 시퀀스 구성


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone
from sklearn.metrics import f1_score, confusion_matrix

DATA_DIR = Path(".")  # 이 Notebook이 들어 있는 problem_04 폴더
CHANNELS = ["force_x", "force_y", "force_z", "torque_x", "torque_y", "torque_z"]
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

def pack_sequences(frame, ordered_ids):
    if set(frame.sample_id) != set(ordered_ids):
        raise ValueError("ID mismatch")
    groups = {key: group.sort_values("step") for key, group in frame.groupby("sample_id")}
    arrays = []
    for key in ordered_ids:
        group = groups[key]
        if not np.array_equal(group.step.to_numpy(), np.arange(15)):
            raise ValueError(f"Invalid steps: {key}")
        values = group[CHANNELS].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"Missing/nonfinite values: {key}")
        arrays.append(values)
    return np.stack(arrays)

X_seq = pack_sequences(train_series, labels.sample_id)
X_test_seq = pack_sequences(test_series, sample.sample_id)
y = labels.label.to_numpy()
print("Train:", X_seq.shape, "Test:", X_test_seq.shape)
print("Label counts:", labels.label.value_counts().sort_index().to_dict())

## 작업 구간 단위 검증 분리


In [ ]:
# sample_id 단위 분리. 동일한 검증 구간으로 모델을 비교하세요.
idx_fit, idx_valid = train_test_split(
    np.arange(len(y)), test_size=0.25, random_state=42, stratify=y
)
assert not set(labels.sample_id.iloc[idx_fit]) & set(labels.sample_id.iloc[idx_valid])
print("Fit / validation segments:", len(idx_fit), len(idx_valid))

## 초기 특징·모델과 검증 — 수정할 부분


In [ ]:
def make_features(sequences):
    # 입력: (작업 구간 수, 15시점, 6채널)
    # 초기 특징: 각 채널의 평균. 이 함수를 수정해 보세요.
    return sequences.mean(axis=1)

X = make_features(X_seq)
X_test = make_features(X_test_seq)
assert X.ndim == 2 and X.shape[0] == len(y)
print("Feature matrix:", X.shape)

# 초기 모델. 다른 모델이나 Pipeline으로 교체해도 됩니다.
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X[idx_fit], y[idx_fit])
valid_pred = model.predict(X[idx_valid])
print("Validation Macro-F1:", f1_score(
    y[idx_valid], valid_pred, labels=list(range(5)), average="macro", zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y[idx_valid], valid_pred, labels=list(range(5))))

## 전체 학습과 최종 CSV 저장


In [ ]:
# 특징 또는 모델을 수정했다면 위 특징 생성·검증 셀을 먼저 다시 실행하세요.
# 아래에서는 model의 설정을 복제하여 전체 train으로 다시 학습합니다.
final_model = clone(model)
final_model.fit(X, y)
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({"sample_id": sample.sample_id, "label": test_pred})
submission.to_csv(DATA_DIR / "submission.csv", index=False, encoding="utf-8-sig")
print("Saved:", (DATA_DIR / "submission.csv").resolve())

## 제출 파일 구조 확인


In [ ]:
# 형식 검사일 뿐 정답 채점이 아닙니다.
check = pd.read_csv(DATA_DIR / "submission.csv")
assert list(check.columns) == ["sample_id", "label"]
assert len(check) == 41 and check.sample_id.is_unique
assert set(check.sample_id) == set(sample.sample_id)
assert pd.api.types.is_integer_dtype(check.label)
assert check.label.isin(range(5)).all()
print("CSV format OK. Notebook도 저장하세요.")